# Approach: SPV-MIA
This approach builds on Milestone 1 and Model 1 by adding SPV-MIA probabilistic
variation features, adapted from Fu et al. (NeurIPS 2024). The 360M model serves
as the target and the 135M model as the reference.

In [1]:
# Core imports
import wandb
from huggingface_hub import HfApi, notebook_login
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM
import re
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
import warnings
warnings.filterwarnings('ignore')

# Set device to use gpu if possible.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Using device: cuda
GPU: NVIDIA H100 80GB HBM3
GPU Memory: 79.2 GB


In [20]:
# Tracking results
# HF
notebook_login()

# W&B login
from google.colab import userdata
wandb.login(key=userdata.get('WB_API_KEY'))

# Initialise a W&B run
wandb.init(
    project="mia-tcob",
    name="smollm2-135m-360m-spv-mia",
)

# Log all hyperparameters and settings for reproducibility
wandb.config.update({
    # Feature selection
    "mi_k": 15,
    # XGBoost
    "xgb_n_estimators": 200,
    "xgb_learning_rate": 0.05,
    "xgb_max_depth": 4,
    "xgb_subsample": 0.8,
    "xgb_colsample_bytree": 0.8,
    # Pipeline
    "batch_size": 500,
    "max_token_length": 512,
    "min_k_values": [10, 20, 30],
    # Models
    "target_model": "UBC-SLIME/colx_531_smollm2-360m",
    "reference_model":"UBC-SLIME/colx_531_smollm2-135m",
    "dataset": "UBC-SLIME/colx_531_group_project",
    # SPV-MIA parameters
    "spv_num_perturbations": 2,
    "spv_mask_pct": 0.3,
    "spv_span_length": 2,
    "spv_mask_model": "t5-base",
})

print("W&B run initialised:", wandb.run.name)
print("W&B project:", wandb.run.project)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


W&B run initialised: smollm2-135m-360m-spv-mia
W&B project: mia-tcob


In [8]:
# Load dataset and models
print("Loading dataset...")
dataset = load_dataset("UBC-SLIME/colx_531_group_project")

print("Loading models...")
model_135m = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-135m")
model_360m = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-360m")
tokenizer = AutoTokenizer.from_pretrained("UBC-SLIME/colx_531_smollm2-135m")

# Move models to device
model_135m.to(device)
model_360m.to(device)
model_135m.eval()
model_360m.eval()

print(f"Dataset: {len(dataset['train'])} train, {len(dataset['validation'])} validation samples")
print("Models loaded: 135M and 360M parameters")

Loading dataset...


train.json:   0%|          | 0.00/102M [00:00<?, ?B/s]

dev.json:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/30.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/110 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataset: 50000 train, 10000 validation samples
Models loaded: 135M and 360M parameters


In [23]:
# Add T5 model for masking / text perturbations

print("--- Loading T5-base for perturbations ---")
# download from HF and load to CPU to free memory for SmolLM2 models
t5_model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")
# load tokenizer for T5
t5_tokenizer = AutoTokenizer.from_pretrained("t5-base")
# set T5 to evaluation mode
t5_model.eval()
print("--- T5-base loaded on CPU ---")

--- Loading T5-base for perturbations ---


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

--- T5-base loaded on CPU ---


# Feature Extraction
Existing features from Model 1 retained, with addition of SPV-MIA probabilistic variation features adapted from Fu et al (NeurIPS 2024).

In [10]:
def compute_perplexity_and_features(model, tokenizer, text, max_length=512):
    """
    Compute perplexity and other features that might indicate memorization.

    Returns:
        dict with perplexity, loss, confidence metrics, Min-K% features

    """
    inputs = tokenizer(text, return_tensors="pt", max_length=max_length,
                       truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        logits = outputs.logits

        # Lower loss, model has likely seen this pattern before
        loss = outputs.loss.item()

        # Low perplexity, more likely seen before
        perplexity = torch.exp(outputs.loss).item()

        shift_logits = logits[..., :-1, :]
        shift_labels = inputs["input_ids"][..., 1:]
        token_probs = F.softmax(shift_logits, dim=-1)

        # Probability of the actual token at each position
        actual_token_probs = token_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)

        # Average confidence across all tokens
        mean_token_prob = actual_token_probs.mean().item()

        # Lowest / highest single-token confidence
        min_token_prob = actual_token_probs.min().item()
        max_token_prob = actual_token_probs.max().item()

        # Lower std, uniformly high confidence, more likely seen before
        std_token_prob = actual_token_probs.std().item()

        # Higher entropy, more uncertain, less likely seen before
        token_entropies = -(token_probs * torch.log(token_probs + 1e-10)).sum(dim=-1)
        mean_entropy = token_entropies.mean().item()

        # High top-k mass, confident and likely seen before
        top_k_probs, _ = torch.topk(token_probs, k=10, dim=-1)
        top_k_mass = top_k_probs.sum(dim=-1).mean().item()

        # Low ratio, few uncertain tokens, likely seen before
        low_conf_tokens = (actual_token_probs < 0.1).sum().item()
        low_conf_ratio = low_conf_tokens / actual_token_probs.shape[1]

        # Min-K% features
        # Average probability of the bottom K% of tokens.
        # Members have higher confidence even on their hardest tokens.
        sorted_probs = actual_token_probs.sort(dim=-1).values  # ascending
        n_tokens = sorted_probs.shape[1]

        k10 = max(1, int(n_tokens * 0.10))
        min_k10_prob = sorted_probs[:, :k10].mean().item()

        k20 = max(1, int(n_tokens * 0.20))
        min_k20_prob = sorted_probs[:, :k20].mean().item()

        k30 = max(1, int(n_tokens * 0.30))
        min_k30_prob = sorted_probs[:, :k30].mean().item()

    result = {
        "perplexity":      perplexity,
        "loss":            loss,
        "mean_token_prob": mean_token_prob,
        "min_token_prob":  min_token_prob,
        "max_token_prob":  max_token_prob,
        "std_token_prob":  std_token_prob,
        "mean_entropy":    mean_entropy,
        "top_k_mass":      top_k_mass,
        "low_conf_ratio":  low_conf_ratio,
        "min_k10_prob":    min_k10_prob,
        "min_k20_prob":    min_k20_prob,
        "min_k30_prob":    min_k30_prob,
        "num_tokens":      n_tokens,
    }

    return result

In [34]:
# SPV-MIA functions - adapted from code provided by authors of
# Fu et al (NeurIPS 2024) paper at:
# https://github.com/tsinghua-fib-lab/NeurIPS2024_SPV-MIA/tree/main
# specifically, attack_model.py, with some support from Claude AI

# ── Step 1 ──────────────────────────────────────────────────────────────
def mask_text(text, mask_pct=0.3, span_length=2):
    """Replace ~30% of word spans with <extra_id_0>, <extra_id_1> etc."""
    words = text.split()

    if len(words) < 4:
        return text, 0

    n_spans = max(1, int(mask_pct * len(words) / (span_length + 2)))

    mask_string = "<<<mask>>>"
    n_masked = 0
    attempts = 0

    while n_masked < n_spans and attempts < 100:
        start = np.random.randint(0, max(1, len(words) - span_length))
        end = start + span_length
        search_start = max(0, start - 1)
        search_end = min(len(words), end + 1)
        if mask_string not in words[search_start:search_end]:
            words[start:end] = [mask_string]
            n_masked += 1
        attempts += 1

    num_masks = 0
    for idx, word in enumerate(words):
        if word == mask_string:
            words[idx] = f"<extra_id_{num_masks}>"
            num_masks += 1

    masked_text = " ".join(words)
    return masked_text, num_masks


# ── Step 2 ──────────────────────────────────────────────────────────────
def fill_masks_with_t5(masked_text, num_masks, n_perturbations=2):
    """Ask T5 to fill in the masked spans and return n filled texts."""

    stop_id = t5_tokenizer.encode(f"<extra_id_{num_masks}>")[0]

    # tokenize on CPU — no .to(device) since T5 is on CPU
    inputs = t5_tokenizer(
        masked_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    # generate all perturbations in one T5 call
    with torch.no_grad():
        outputs = t5_model.generate(
            **inputs,
            max_length=300,
            do_sample=True,
            top_p=1.0,
            num_return_sequences=n_perturbations,
            eos_token_id=stop_id
        )

    # decode all outputs and return as a list
    filled_texts = [
        t5_tokenizer.decode(output, skip_special_tokens=False)
        .replace("<pad>", "").replace("</s>", "").strip()
        for output in outputs
    ]
    return filled_texts


# ── Step 3 ──────────────────────────────────────────────────────────────
def reconstruct_text(masked_text, t5_output, num_masks):
    """Replace mask tags in the original text with T5's suggestions."""
    pattern = re.compile(r"<extra_id_\d+>")

    suggested_fills = [w.strip() for w in pattern.split(t5_output)[1:-1]]
    words = masked_text.split(" ")

    if len(suggested_fills) >= num_masks:
        for fill_idx in range(num_masks):
            tag = f"<extra_id_{fill_idx}>"
            if tag in words:
                words[words.index(tag)] = suggested_fills[fill_idx]
        return " ".join(words)
    else:
        print(f"Reconstruction failed: T5 gave {len(suggested_fills)} fills but need {num_masks}")
        return None


# ── Main perturbation function ───────────────────────────────────────────
def generate_perturbations(text, n_perturbations=2):
    """
    Generate n slightly modified versions of the text.
    Each version has ~30% of its words replaced by T5.
    """
    try:
        # Step 1 — mask part of the text once
        masked_text, num_masks = mask_text(text)

        if num_masks == 0:
            return [text] * n_perturbations

        # Step 2 — ask T5 to fill in the blanks, getting n outputs in one call
        t5_outputs = fill_masks_with_t5(masked_text, num_masks, n_perturbations)

        # Step 3 — reconstruct each perturbed text from T5's outputs
        perturbed_texts = []
        for t5_output in t5_outputs:
            perturbed = reconstruct_text(masked_text, t5_output, num_masks)
            if perturbed and perturbed.strip():
                perturbed_texts.append(perturbed)
            else:
                perturbed_texts.append(text)

        return perturbed_texts

    except Exception as e:
        print(f"Perturbation failed: {str(e)[:80]}")
        return [text] * n_perturbations


# ── SPV feature computation ──────────────────────────────────────────────
def compute_spv_features(text, original_loss_360m=None,
                         original_loss_135m=None,
                         n_perturbations=2, max_length=512):
    """
    Compute SPV-MIA features for a single text.
    Accepts pre-computed original losses to avoid redundant forward passes.
    Uses 360M as target model and 135M as reference model.
    Returns spv_target, spv_reference, spv_calibrated.
    """

    def get_loss(model, text):
        """Get the model's loss on a single text."""
        inputs = tokenizer(
            text,
            return_tensors="pt",
            max_length=max_length,
            truncation=True,
            padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():  # Option 2 — faster than no_grad
            outputs = model(**inputs, labels=inputs["input_ids"])
        return outputs.loss.item()

    try:
        # Option 1 — use pre-computed losses if available
        # saves 2 forward passes per text
        if original_loss_360m is None:
            original_loss_360m = get_loss(model_360m, text)
        if original_loss_135m is None:
            original_loss_135m = get_loss(model_135m, text)

        # generate perturbed versions of the text
        perturbed_texts = generate_perturbations(text, n_perturbations)

        # get loss on each perturbed version from both models
        perturbed_losses_360m = [get_loss(model_360m, p) for p in perturbed_texts]
        perturbed_losses_135m = [get_loss(model_135m, p) for p in perturbed_texts]

        # compute the average perturbed loss for each model
        mean_perturbed_360m = np.mean(perturbed_losses_360m)
        mean_perturbed_135m = np.mean(perturbed_losses_135m)

        # SPV = how much does loss increase when we perturb the text?
        # large SPV = steep probability peak = likely memorized = likely member
        spv_target    = mean_perturbed_360m - original_loss_360m
        spv_reference = mean_perturbed_135m - original_loss_135m

        # calibrated signal = remove natural text complexity
        # leaving only the memorization signal
        spv_calibrated = spv_target - spv_reference

        return {
            "spv_target":     spv_target,
            "spv_reference":  spv_reference,
            "spv_calibrated": spv_calibrated,
        }

    except Exception as e:
        print(f"SPV computation failed: {str(e)[:100]}")
        return {
            "spv_target":     0.0,
            "spv_reference":  0.0,
            "spv_calibrated": 0.0,
        }

In [12]:
def analyze_sample_batch(model, tokenizer, texts, labels, model_name="model"):
    """
    Analyze a batch of texts and return features for membership inference.

    """
    results = []

    for text, label in tqdm(zip(texts, labels), total=len(texts),
                            desc=f"Analyzing with {model_name}"):
        try:
            features = compute_perplexity_and_features(
                model, tokenizer, text
            )
            features["label"] = label
            features["text_length"] = len(text)
            results.append(features)
        except Exception as e:
            print(f"Error processing text (len={len(text)}): {str(e)[:100]}")
            results.append({
                "perplexity": float("inf"),
                "loss": float("inf"),
                "mean_token_prob": 0,
                "min_k10_prob": 0,
                "min_k20_prob": 0,
                "min_k30_prob": 0,
                "label": label,
                "text_length": len(text),
            })

    return pd.DataFrame(results)

In [13]:
def process_dataset_in_batches(model, tokenizer, texts, labels,
                               model_name="model", batch_size=1000):
    """
    Process large dataset in batches for memory efficiency.
    """
    all_results = []
    print(f"Processing {len(texts)} samples in batches of {batch_size}...")

    for i in range(0, len(texts), batch_size):
        end_idx     = min(i + batch_size, len(texts))
        batch_texts  = texts[i:end_idx]
        batch_labels = labels[i:end_idx]

        print(f"Batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}: "
              f"{end_idx - i} samples")

        batch_results = analyze_sample_batch(
            model, tokenizer, batch_texts, batch_labels,
            model_name
        )
        all_results.append(batch_results)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_results = pd.concat(all_results, ignore_index=True)
    print(f"Completed processing {len(final_results)} samples with {model_name}")
    return final_results

In [35]:
def compute_spv_for_dataset(texts, labels, df_135m=None,
                             df_360m=None, batch_size=500,
                             save_dir=None):
    """
    Compute SPV-MIA features for all texts.
    Accepts pre-computed feature dataframes to reuse existing loss values.
    Saves each batch to Drive if save_dir is provided.
    Returns a DataFrame with spv_target, spv_reference, spv_calibrated, and label columns.
    """
    all_results = []
    print(f"Computing SPV features for {len(texts)} samples...")

    for i in range(0, len(texts), batch_size):
        end_idx       = min(i + batch_size, len(texts))
        batch_texts   = texts[i:end_idx]
        batch_labels  = labels[i:end_idx]
        batch_num     = i//batch_size + 1
        total_batches = (len(texts)-1)//batch_size + 1

        print(f"\nSPV Batch {batch_num}/{total_batches} ({end_idx - i} samples)")

        batch_results = []
        for j, (text, label) in enumerate(tqdm(zip(batch_texts, batch_labels),
                                               total=len(batch_texts),
                                               desc="Computing SPV features")):
            # Option 1 — reuse pre-computed losses if available
            global_idx = i + j
            loss_360m = df_360m['loss'].iloc[global_idx] if df_360m is not None else None
            loss_135m = df_135m['loss'].iloc[global_idx] if df_135m is not None else None

            spv_features = compute_spv_features(
                text,
                original_loss_360m=loss_360m,
                original_loss_135m=loss_135m
            )
            spv_features["label"] = label
            batch_results.append(spv_features)

        batch_df = pd.DataFrame(batch_results)
        all_results.append(batch_df)

        # save each batch to Drive immediately after completion
        if save_dir is not None:
            batch_path = f"{save_dir}spv_batch_{batch_num}.csv"
            batch_df.to_csv(batch_path, index=False)
            print(f"Batch {batch_num} saved to Drive: {batch_path}")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_results = pd.concat(all_results, ignore_index=True)
    print(f"\nSPV feature computation complete: {len(final_results)} samples")
    return final_results

# Dataset Preparation
Load and prepare the full training and validation datasets

In [15]:
# Load full dataset splits
train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()

print(f"Training set: {len(train_df)} samples")
print(f"Validation set: {len(val_df)} samples")
print(f"Test set: {len(test_df)} samples")
print(f"Training - Members: {sum(train_df['is_member'])}, Non-members: {len(train_df) - sum(train_df['is_member'])}")
print(f"Validation - Members: {sum(val_df['is_member'])}, Non-members: {len(val_df) - sum(val_df['is_member'])}")

# Extract texts and labels
train_texts = train_df['text'].tolist()
train_labels = train_df['is_member'].tolist()
val_texts = val_df['text'].tolist()
val_labels = val_df['is_member'].tolist()
test_texts = test_df['text'].tolist()
test_labels = [0] * len(test_texts)  # Dummy labels for test set

Training set: 50000 samples
Validation set: 10000 samples
Test set: 15000 samples
Training - Members: 25000, Non-members: 25000
Validation - Members: 5000, Non-members: 5000


In [16]:
# Process training data
print("Processing training data with 135M model...")
train_results_135m = process_dataset_in_batches(
    model_135m, tokenizer, train_texts, train_labels, "135M", batch_size=500
)

print("\nProcessing training data with 360M model...")
train_results_360m = process_dataset_in_batches(
    model_360m, tokenizer, train_texts, train_labels, "360M", batch_size=500
)

print(f"\nTraining data processing complete!")
print(f"135M results: {train_results_135m.shape}")
print(f"360M results: {train_results_360m.shape}")

Processing training data with 135M model...
Processing 50000 samples in batches of 500...
Batch 1/100: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:11<00:00, 42.79it/s]


Batch 2/100: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:09<00:00, 50.76it/s]


Batch 3/100: 500 samples


...

Batch 99/100: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:10<00:00, 48.56it/s]


Batch 100/100: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:10<00:00, 48.08it/s]

Completed processing 50000 samples with 360M

Training data processing complete!
135M results: (50000, 15)
360M results: (50000, 15)


In [17]:
# Process validation data
print("Processing validation data with 135M model...")
val_results_135m = process_dataset_in_batches(
    model_135m, tokenizer, val_texts, val_labels, "135M", batch_size=500
)

print("\nProcessing validation data with 360M model...")
val_results_360m = process_dataset_in_batches(
    model_360m, tokenizer, val_texts, val_labels, "360M", batch_size=500
)

print(f"\nValidation data processing complete!")
print(f"135M results: {val_results_135m.shape}")
print(f"360M results: {val_results_360m.shape}")

Processing validation data with 135M model...
Processing 10000 samples in batches of 500...
Batch 1/20: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:09<00:00, 50.91it/s]


Batch 2/20: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:09<00:00, 51.15it/s]


Batch 3/20: 500 samples


...

Batch 19/20: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:10<00:00, 48.47it/s]


Batch 20/20: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:10<00:00, 48.45it/s]

Completed processing 10000 samples with 360M

Validation data processing complete!
135M results: (10000, 15)
360M results: (10000, 15)


In [18]:
# Process test data
print("Processing test data with 135M model...")
test_results_135m = process_dataset_in_batches(
    model_135m, tokenizer, test_texts, test_labels, "135M", batch_size=500
)

print("\nProcessing test data with 360M model...")
test_results_360m = process_dataset_in_batches(
    model_360m, tokenizer, test_texts, test_labels, "360M", batch_size=500
)

print(f"\nTest data processing complete!")
print(f"135M results: {test_results_135m.shape}")
print(f"360M results: {test_results_360m.shape}")

Processing test data with 135M model...
Processing 15000 samples in batches of 500...
Batch 1/30: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:09<00:00, 51.38it/s]


Batch 2/30: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:09<00:00, 52.06it/s]


Batch 3/30: 500 samples


...

Batch 29/30: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:10<00:00, 49.07it/s]


Batch 30/30: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:10<00:00, 48.71it/s]

Completed processing 15000 samples with 360M

Test data processing complete!
135M results: (15000, 15)
360M results: (15000, 15)


In [37]:
# Compute SPV-MIA features for a subset of each split
# Full dataset too expensive — SPV is computationally intensive
# (2 model forward passes × 2 perturbations + T5 per text)

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/mia_features/"

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory ready: {SAVE_DIR}")

N_TRAIN = 3000
N_VAL   = 500
N_TEST  = 500

print(f"Computing SPV features for {N_TRAIN} training samples...")
train_spv = compute_spv_for_dataset(
    train_texts[:N_TRAIN],
    train_labels[:N_TRAIN],
    df_135m=train_results_135m.iloc[:N_TRAIN].reset_index(drop=True),
    df_360m=train_results_360m.iloc[:N_TRAIN].reset_index(drop=True),
    save_dir=SAVE_DIR
)

print(f"\nComputing SPV features for {N_VAL} validation samples...")
val_spv = compute_spv_for_dataset(
    val_texts[:N_VAL],
    val_labels[:N_VAL],
    df_135m=val_results_135m.iloc[:N_VAL].reset_index(drop=True),
    df_360m=val_results_360m.iloc[:N_VAL].reset_index(drop=True),
    save_dir=SAVE_DIR
)

print(f"\nComputing SPV features for {N_TEST} test samples...")
test_spv = compute_spv_for_dataset(
    test_texts[:N_TEST],
    test_labels[:N_TEST],
    df_135m=test_results_135m.iloc[:N_TEST].reset_index(drop=True),
    df_360m=test_results_360m.iloc[:N_TEST].reset_index(drop=True),
    save_dir=SAVE_DIR
)

print("\nAll SPV features computed!")
print(f"Train SPV shape:      {train_spv.shape}")
print(f"Validation SPV shape: {val_spv.shape}")
print(f"Test SPV shape:       {test_spv.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save directory ready: /content/drive/MyDrive/mia_features/
Computing SPV features for 3000 training samples...
Computing SPV features for 3000 samples...

SPV Batch 1/6 (500 samples)


Computing SPV features:   1%|          | 3/500 [00:07<24:04,  2.91s/it]

Reconstruction failed: T5 gave 27 fills but need 33
Reconstruction failed: T5 gave 27 fills but need 33


Computing SPV features:   1%|          | 5/500 [00:13<25:37,  3.11s/it]

Reconstruction failed: T5 gave 27 fills but need 29
Reconstruction failed: T5 gave 27 fills but need 29


...

Reconstruction failed: T5 gave 22 fills but need 31
Reconstruction failed: T5 gave 22 fills but need 31


Computing SPV features:  93%|█████████▎| 463/500 [13:55<01:25,  2.31s/it]

Reconstruction failed: T5 gave 27 fills but need 29
Reconstruction failed: T5 gave 27 fills but need 29


Computing SPV features: 100%|██████████| 500/500 [14:47<00:00,  1.77s/it]

Batch 1 saved to Drive: /content/drive/MyDrive/mia_features/spv_batch_1.csv

SPV feature computation complete: 500 samples

All SPV features computed!
Train SPV shape:      (3000, 4)
Validation SPV shape: (500, 4)
Test SPV shape:       (500, 4)


In [38]:
# Save all extracted features to disk immediately after extraction
# This makes feature extraction and classifier training fully independent
# sessions.

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/mia_features/"

import os
os.makedirs(SAVE_DIR, exist_ok=True)

# existing features
train_results_135m.to_csv(f"{SAVE_DIR}train_features_135m.csv", index=False)
train_results_360m.to_csv(f"{SAVE_DIR}train_features_360m.csv", index=False)
val_results_135m.to_csv(  f"{SAVE_DIR}val_features_135m.csv",   index=False)
val_results_360m.to_csv(  f"{SAVE_DIR}val_features_360m.csv",   index=False)
test_results_135m.to_csv( f"{SAVE_DIR}test_features_135m.csv",  index=False)
test_results_360m.to_csv( f"{SAVE_DIR}test_features_360m.csv",  index=False)

# SPV features
train_spv.to_csv(f"{SAVE_DIR}train_spv.csv", index=False)
val_spv.to_csv(  f"{SAVE_DIR}val_spv.csv",   index=False)
test_spv.to_csv( f"{SAVE_DIR}test_spv.csv",  index=False)

print(f"All features saved to {SAVE_DIR}")
print(f"Files saved:")
print(f"  - train_features_135m.csv, train_features_360m.csv")
print(f"  - val_features_135m.csv,   val_features_360m.csv")
print(f"  - test_features_135m.csv,  test_features_360m.csv")
print(f"  - train_spv.csv, val_spv.csv, test_spv.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All features saved to /content/drive/MyDrive/mia_features/
Files saved:
  - train_features_135m.csv, train_features_360m.csv
  - val_features_135m.csv,   val_features_360m.csv
  - test_features_135m.csv,  test_features_360m.csv
  - train_spv.csv, val_spv.csv, test_spv.csv


# Membership Inference Classifier

In [ ]:
# Load pre-extracted features
# Only needed when starting a fresh session without re-running extraction.
# Comment this block out if running the full pipeline top to bottom.

# from google.colab import drive
# drive.mount('/content/drive')
# SAVE_DIR = "/content/drive/MyDrive/mia_features/"

# # existing features
# train_results_135m = pd.read_csv(f"{SAVE_DIR}train_features_135m.csv")
# train_results_360m = pd.read_csv(f"{SAVE_DIR}train_features_360m.csv")
# val_results_135m = pd.read_csv(f"{SAVE_DIR}val_features_135m.csv")
# val_results_360m = pd.read_csv(f"{SAVE_DIR}val_features_360m.csv")
# test_results_135m = pd.read_csv(f"{SAVE_DIR}test_features_135m.csv")
# test_results_360m = pd.read_csv(f"{SAVE_DIR}test_features_360m.csv")

# # SPV features
# train_spv = pd.read_csv(f"{SAVE_DIR}train_spv.csv")
# val_spv   = pd.read_csv(f"{SAVE_DIR}val_spv.csv")
# test_spv  = pd.read_csv(f"{SAVE_DIR}test_spv.csv")


# print("Features loaded from disk.")
# print(f"  train_135m: {train_results_135m.shape}")
# print(f"  train_360m: {train_results_360m.shape}")
# print(f"  val_135m:   {val_results_135m.shape}")
# print(f"  val_360m:   {val_results_360m.shape}")
# print(f"  train_spv:  {train_spv.shape}")
# print(f"  val_spv:    {val_spv.shape}")

In [39]:
def build_membership_classifier(train_135m, train_360m, val_135m, val_360m,
                                train_spv, val_spv):
    """
    Build classifier using full training set and evaluate on validation set.
    SPV features computed on a subset - remaining samples filled with zeros to
    indicate no SPV signal.

    """
    print("Building membership inference classifier...")

    # Feature columns
    feature_cols = ['perplexity', 'loss', 'mean_token_prob', 'min_token_prob', 'max_token_prob',
                    'std_token_prob', 'mean_entropy', 'top_k_mass', 'low_conf_ratio',
                    'min_k10_prob', 'min_k20_prob', 'min_k30_prob']

    # Prepare training features
    train_features_135m = train_135m[feature_cols].copy()
    train_features_360m = train_360m[feature_cols].copy()
    train_features_135m.columns = [f"{col}_135m" for col in train_features_135m.columns]
    train_features_360m.columns = [f"{col}_360m" for col in train_features_360m.columns]
    X_train = pd.concat([train_features_135m, train_features_360m], axis=1)
    y_train = train_135m['label']

    # Prepare validation features
    val_features_135m = val_135m[feature_cols].copy()
    val_features_360m = val_360m[feature_cols].copy()
    val_features_135m.columns = [f"{col}_135m" for col in val_features_135m.columns]
    val_features_360m.columns = [f"{col}_360m" for col in val_features_360m.columns]
    X_val = pd.concat([val_features_135m, val_features_360m], axis=1)
    y_val = val_135m['label']

    # Cross-model difference and ratio features
    # capture how much more confident the 360m model is vs the 135m model
    # since previous runs suggest 360m model has stronger features
    diff_cols = ['loss', 'perplexity', 'min_k10_prob',
                'min_k20_prob', 'min_k30_prob', 'min_token_prob']

    for col in diff_cols:
        X_train[f'{col}_diff']  = X_train[f'{col}_360m'] - X_train[f'{col}_135m']
        X_train[f'{col}_ratio'] = X_train[f'{col}_360m'] / (X_train[f'{col}_135m'] + 1e-10)
        X_val[f'{col}_diff']    = X_val[f'{col}_360m']   - X_val[f'{col}_135m']
        X_val[f'{col}_ratio']   = X_val[f'{col}_360m']   / (X_val[f'{col}_135m'] + 1e-10)

    print(f"Features after adding cross-model diffs: {X_train.shape[1]}")

    # Add SPV-MIA features
    # SPV computed on 3,000 subset — fill remaining samples with 0.0
    # XGBoost will learn that 0.0 = no SPV signal available
    n_train_full = len(train_135m)
    n_val_full   = len(val_135m)
    n_spv_train  = len(train_spv)
    n_spv_val    = len(val_spv)

    print(f"SPV features available for {n_spv_train}/{n_train_full} train samples")
    print(f"SPV features available for {n_spv_val}/{n_val_full} val samples")

    # create full-size SPV arrays filled with zeros
    spv_train_full = pd.DataFrame({
        'spv_target':     [0.0] * n_train_full,
        'spv_reference':  [0.0] * n_train_full,
        'spv_calibrated': [0.0] * n_train_full,
    })
    spv_val_full = pd.DataFrame({
        'spv_target':     [0.0] * n_val_full,
        'spv_reference':  [0.0] * n_val_full,
        'spv_calibrated': [0.0] * n_val_full,
    })

    # fill in real SPV values for the subset
    spv_train_full.iloc[:n_spv_train] = train_spv[['spv_target', 'spv_reference', 'spv_calibrated']].values
    spv_val_full.iloc[:n_spv_val]     = val_spv[['spv_target', 'spv_reference', 'spv_calibrated']].values

    # add to feature matrices
    X_train['spv_target']     = spv_train_full['spv_target'].values
    X_train['spv_reference']  = spv_train_full['spv_reference'].values
    X_train['spv_calibrated'] = spv_train_full['spv_calibrated'].values

    X_val['spv_target']     = spv_val_full['spv_target'].values
    X_val['spv_reference']  = spv_val_full['spv_reference'].values
    X_val['spv_calibrated'] = spv_val_full['spv_calibrated'].values

    print(f"Features after adding SPV: {X_train.shape[1]}")

    # Clean up inf / NaN
    train_medians = X_train.replace([np.inf, -np.inf], np.nan).median()
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(train_medians)
    X_val = X_val.replace([np.inf, -np.inf], np.nan).fillna(train_medians)

    X_train = X_train.fillna(0)
    X_val = X_val.fillna(0)

    all_feature_names = X_train.columns.tolist()
    print(f"Total features: {X_train.shape[1]}")

    selected_features = all_feature_names

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    print(f"\nTraining set:   {X_train.shape[0]} samples")
    print(f"Validation set: {X_val.shape[0]} samples")

    # Train classifiers
    # logistic regression, random forest, XGBoost
    classifiers = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=42,
        ),
    }

    results = {}

    for name, clf in classifiers.items():
        print(f"\nTraining {name}...")

        # Logistic Regression requires scaled features
        # Random Forest and XGBoost operate on raw feature values
        if name == 'Logistic Regression':
            clf.fit(X_train_scaled, y_train)
            y_pred_proba = clf.predict_proba(X_val_scaled)[:, 1]
        else:
            clf.fit(X_train, y_train)
            y_pred_proba = clf.predict_proba(X_val)[:, 1]

        auc = roc_auc_score(y_val, y_pred_proba)

        fpr, tpr, thresholds = roc_curve(y_val, y_pred_proba)
        idx = np.argmin(np.abs(fpr - 0.1))
        tpr_at_fpr_01 = tpr[idx]
        actual_fpr    = fpr[idx]

        y_pred   = (y_pred_proba > 0.5).astype(int)
        accuracy = (y_pred == y_val).mean()

        print(f"  AUC:           {auc:.4f}")
        print(f"  TPR@FPR=0.1:   {tpr_at_fpr_01:.4f}  (actual FPR: {actual_fpr:.4f})")
        print(f"  Accuracy:      {accuracy:.4f}")

        results[name] = {
            'classifier': clf,
            'scaler': scaler if name == 'Logistic Regression' else None,
            'auc': auc,
            'tpr_at_fpr_01': tpr_at_fpr_01,
            'accuracy': accuracy,
            'feature_names': selected_features
        }

        # W&B log classifier metrics
        wandb.log({
            f"{name}/auc":           auc,
            f"{name}/tpr_at_fpr_01": tpr_at_fpr_01,
            f"{name}/accuracy":      accuracy,
        })

        # Feature importances — available for both tree-based models
        if name in ('Random Forest', 'XGBoost'):
            importance_df = pd.DataFrame({
                'feature':    selected_features,
                'importance': clf.feature_importances_
            }).sort_values('importance', ascending=False)

            print(f"\n  Top 10 Most Important Features ({name}, post-selection):")
            for _, row in importance_df.head(10).iterrows():
                print(f"    {row['feature']}: {row['importance']:.4f}")

            # W&B: log feature importances as a bar chart
            wandb.log({
                f"{name}/feature_importances": wandb.plot.bar(
                    wandb.Table(dataframe=importance_df),
                    label="feature",
                    value="importance",
                    title=f"{name} Feature Importances",
                )
            })

    return results

In [40]:
classifier_results = build_membership_classifier(
    train_results_135m, train_results_360m,
    val_results_135m,   val_results_360m,
    train_spv, val_spv
)

Building membership inference classifier...
Features after adding cross-model diffs: 36
SPV features available for 3000/50000 train samples
SPV features available for 500/10000 val samples
Features after adding SPV: 39
Total features: 39

Training set:   50000 samples
Validation set: 10000 samples

Training Logistic Regression...
  AUC:           0.8903
  TPR@FPR=0.1:   0.6640  (actual FPR: 0.1000)
  Accuracy:      0.8216

Training Random Forest...
  AUC:           0.8863
  TPR@FPR=0.1:   0.6538  (actual FPR: 0.1002)
  Accuracy:      0.8075

  Top 10 Most Important Features (Random Forest, post-selection):
    loss_ratio: 0.1401
    loss_diff: 0.0589
    min_k20_prob_diff: 0.0521
    min_k30_prob_diff: 0.0521
    min_k10_prob_diff: 0.0520
    perplexity_ratio: 0.0518
    min_token_prob_360m: 0.0333
    min_k10_prob_360m: 0.0321
    min_k30_prob_ratio: 0.0308
    mean_entropy_360m: 0.0286

Training XGBoost...
  AUC:           0.8955
  TPR@FPR=0.1:   0.6662  (actual FPR: 0.1004)
  Accura

# Results Summary

In [41]:
# Results summary
print("MEMBERSHIP INFERENCE ATTACK RESULTS")
print("=" * 50)

for name, results in classifier_results.items():
    print(f"\n{name}:")
    print(f"  AUC: {results['auc']:.4f}")
    print(f"  TPR@FPR=0.1: {results['tpr_at_fpr_01']:.4f}")
    print(f"  Accuracy: {results['accuracy']:.4f}")

# Best model
best_model = max(classifier_results.values(), key=lambda x: x['auc'])
print(f"\nBEST MODEL PERFORMANCE:")
print(f"AUC: {best_model['auc']:.4f}")
print(f"TPR@FPR=0.1: {best_model['tpr_at_fpr_01']:.4f}")
print(f"Accuracy: {best_model['accuracy']:.4f}")

print(f"\nDataset processed:")
print(f"Training samples: {len(train_results_135m)}")
print(f"Validation samples: {len(val_results_135m)}")
print(f"Total features: {len(best_model['feature_names'])}")

# W&B log best-model summary metrics
best_classifier_name = max(classifier_results.keys(),
                           key=lambda x: classifier_results[x]['auc'])
wandb.summary["best_classifier"] = best_classifier_name
wandb.summary["best_auc"] = best_model['auc']
wandb.summary["best_tpr_at_fpr_01"] = best_model['tpr_at_fpr_01']
wandb.summary["best_accuracy"] = best_model['accuracy']
wandb.summary["n_train_samples"] = len(train_results_135m)
wandb.summary["n_val_samples"] = len(val_results_135m)
wandb.summary["n_features_selected"]= len(best_model['feature_names'])
wandb.summary["selected_features"]  = best_model['feature_names']

MEMBERSHIP INFERENCE ATTACK RESULTS

Logistic Regression:
  AUC: 0.8903
  TPR@FPR=0.1: 0.6640
  Accuracy: 0.8216

Random Forest:
  AUC: 0.8863
  TPR@FPR=0.1: 0.6538
  Accuracy: 0.8075

XGBoost:
  AUC: 0.8955
  TPR@FPR=0.1: 0.6662
  Accuracy: 0.8110

BEST MODEL PERFORMANCE:
AUC: 0.8955
TPR@FPR=0.1: 0.6662
Accuracy: 0.8110

Dataset processed:
Training samples: 50000
Validation samples: 10000
Total features: 39


In [42]:
# Save results and models
import pickle

# Save the best classifier
best_classifier_name = max(classifier_results.keys(), key=lambda x: classifier_results[x]['auc'])
best_classifier = classifier_results[best_classifier_name]

# Save to files
with open('membership_classifier.pkl', 'wb') as f:
    pickle.dump(best_classifier, f)

# Save feature results
train_results_135m.to_csv('train_features_135m.csv', index=False)
train_results_360m.to_csv('train_features_360m.csv', index=False)
val_results_135m.to_csv('val_features_135m.csv', index=False)
val_results_360m.to_csv('val_features_360m.csv', index=False)

# Save SPV features
train_spv.to_csv('train_spv.csv', index=False)
val_spv.to_csv('val_spv.csv', index=False)
test_spv.to_csv('test_spv.csv', index=False)

print("Results saved:")
print("- membership_classifier.pkl")
print("- train_features_135m.csv")
print("- train_features_360m.csv")
print("- val_features_135m.csv")
print("- val_features_360m.csv")
print("- train_spv.csv")
print("- val_spv.csv")
print("- test_spv.csv")

# W&B: save artefacts so they are versioned alongside the run
artifact = wandb.Artifact(name="membership-classifier", type="model")
artifact.add_file("membership_classifier.pkl")
wandb.log_artifact(artifact)
print("Classifier uploaded to W&B as artifact.")

# HuggingFace Hub: upload classifier and feature CSVs
HF_REPO_ID = "rdj-034/mia-medical-llm"

api = HfApi()
api.create_repo("rdj-034/mia-medical-llm", repo_type="model", exist_ok=True)

# Upload trained classifier
api.upload_file(
    path_or_fileobj="membership_classifier.pkl",
    path_in_repo="membership_classifier.pkl",
    repo_id=HF_REPO_ID,
    repo_type="model",
)

# Upload feature CSVs for reproducibility
for fname in ["train_features_135m.csv", "train_features_360m.csv",
              "val_features_135m.csv",   "val_features_360m.csv",
              "train_spv.csv", "val_spv.csv"]:
    api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=f"features/{fname}",
        repo_id=HF_REPO_ID,
        repo_type="model",
    )

print(f"Classifier and features uploaded to HuggingFace: {HF_REPO_ID}")

Results saved:
- membership_classifier.pkl
- train_features_135m.csv
- train_features_360m.csv
- val_features_135m.csv
- val_features_360m.csv
- train_spv.csv
- val_spv.csv
- test_spv.csv
Classifier uploaded to W&B as artifact.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  membership_classifier.pkl   : 100%|##########|  345kB /  345kB            

Classifier and features uploaded to HuggingFace: rdj-034/mia-medical-llm


# Generate Test Predictions

In [43]:
def prepare_test_predictions(test_135m, test_360m, best_classifier, test_spv):
    """
    Prepare test features and generate predictions for Kaggle submission.
    SPV features computed on a subset — remaining samples filled with zeros.
    """
    print("Preparing test features for prediction...")

    # Feature columns
    feature_cols = ['perplexity', 'loss', 'mean_token_prob', 'min_token_prob', 'max_token_prob',
                   'std_token_prob', 'mean_entropy', 'top_k_mass', 'low_conf_ratio',
                   'min_k10_prob', 'min_k20_prob', 'min_k30_prob']

    # Extract features
    test_features_135m = test_135m[feature_cols].copy()
    test_features_360m = test_360m[feature_cols].copy()

    # Add model suffix
    test_features_135m.columns = [f"{col}_135m" for col in test_features_135m.columns]
    test_features_360m.columns = [f"{col}_360m" for col in test_features_360m.columns]

    # Combine features
    X_test = pd.concat([test_features_135m, test_features_360m], axis=1)

    # cross-model diff columns
    diff_cols = ['loss', 'perplexity', 'min_k10_prob',
             'min_k20_prob', 'min_k30_prob', 'min_token_prob']

    for col in diff_cols:
        X_test[f'{col}_diff']  = X_test[f'{col}_360m'] - X_test[f'{col}_135m']
        X_test[f'{col}_ratio'] = X_test[f'{col}_360m'] / (X_test[f'{col}_135m'] + 1e-10)

    # Option C — SPV features for test set
    # fill full test set with zeros, then fill in real SPV values for subset
    n_test_full = len(test_135m)
    n_spv_test  = len(test_spv)

    print(f"SPV features available for {n_spv_test}/{n_test_full} test samples")

    spv_test_full = pd.DataFrame({
        'spv_target':     [0.0] * n_test_full,
        'spv_reference':  [0.0] * n_test_full,
        'spv_calibrated': [0.0] * n_test_full,
    })
    spv_test_full.iloc[:n_spv_test] = test_spv[['spv_target', 'spv_reference', 'spv_calibrated']].values

    X_test['spv_target']     = spv_test_full['spv_target'].values
    X_test['spv_reference']  = spv_test_full['spv_reference'].values
    X_test['spv_calibrated'] = spv_test_full['spv_calibrated'].values

    # Handle infinite/NaN values
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(X_test.median())

    # Keep only features selected during training
    X_test = X_test[best_classifier['feature_names']]

    # Apply scaling if the classifier uses it
    if best_classifier['scaler'] is not None:
        X_test = best_classifier['scaler'].transform(X_test)

    # Generate predictions
    print("Generating predictions on test set...")
    test_predictions = best_classifier['classifier'].predict_proba(X_test)[:, 1]

    return test_predictions


# Get best classifier
best_classifier_name = max(classifier_results.keys(), key=lambda x: classifier_results[x]['auc'])
best_classifier = classifier_results[best_classifier_name]

print(f"Using {best_classifier_name} for test predictions (AUC: {best_classifier['auc']:.4f})")

# Generate test predictions
test_predictions = prepare_test_predictions(test_results_135m, test_results_360m,
                                            best_classifier, test_spv)

print(f"Generated {len(test_predictions)} predictions")
print(f"Prediction range: [{test_predictions.min():.4f}, {test_predictions.max():.4f}]")
print(f"Mean prediction: {test_predictions.mean():.4f}")

Using XGBoost for test predictions (AUC: 0.8955)
Preparing test features for prediction...
SPV features available for 500/15000 test samples
Generating predictions on test set...
Generated 15000 predictions
Prediction range: [0.0018, 0.9766]
Mean prediction: 0.4990


In [44]:
# Create Kaggle submission file with correct IDs
print("Creating Kaggle submission file...")

# Use the  'id' column from test dataset
test_ids = test_df['id'].tolist()

# Create submission DataFrame with proper ID mapping
submission_df = pd.DataFrame({
    'id': test_ids,
    'is_member': test_predictions
})

# Save submission file
submission_filename = 'membership_inference_submission.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"Kaggle submission saved: {submission_filename}")
print(f"Shape: {submission_df.shape}")
print(f"ID range: {submission_df['id'].min()} to {submission_df['id'].max()}")
print(f"Prediction range: [{submission_df['is_member'].min():.4f}, {submission_df['is_member'].max():.4f}]")

print("\nFirst 5 rows:")
print(submission_df.head())


# W&B: close the run
wandb.finish()
print("W&B run closed.")

Creating Kaggle submission file...
Kaggle submission saved: membership_inference_submission.csv
Shape: (15000, 2)
ID range: 130000 to 37499
Prediction range: [0.0018, 0.9766]

First 5 rows:
       id  is_member
0   32656   0.945469
1  137254   0.038639
2  134884   0.804643
3   33065   0.728444
4   34304   0.879401


Logistic Regression/accuracy,▁
Logistic Regression/auc,▁
Logistic Regression/tpr_at_fpr_01,▁
Random Forest/accuracy,▁
Random Forest/auc,▁
Random Forest/tpr_at_fpr_01,▁
XGBoost/accuracy,▁
XGBoost/auc,▁
XGBoost/tpr_at_fpr_01,▁
Logistic Regression/accuracy,0.8216
Logistic Regression/auc,0.89032


W&B run closed.


In [46]:
from google.colab import files
files.download('membership_inference_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>